In [2]:
from astropy.table import Table
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import pandas as pd
from astropy.wcs import WCS
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_sun
import astropy.units as u
from scipy.signal import savgol_filter
from datetime import datetime
import shutil
import os
import glob
import re


In [5]:

class GradPipeline:
    """
    Pipeline simple pour les données GRAD-300
    """
    def __init__(self, base_path="./GRAD-300/GRAD-300"):
        self.base_path = base_path
        self.location = EarthLocation(lat=43.933*u.deg, lon=5.7153*u.deg, height=654.8*u.m)
        self.file_pattern = r'(\d{8})-(\d{6})_([A-Z]+)-([A-Z0-9]+)-([A-Z0-9]+)_\d+#_\d+#\.fits'
        
        # Constantes pour le spectre
        self.basefreq = 1.398e9
        self.bandwidth = 62.5e6
        self.nchan = 1024
        
        # self.center_ra = 0  # RA du centre en degrés (à ajuster)
        # self.center_dec = 0  # Dec du centre en degrés (à ajuster)
        self.pixel_scale = 0.1 # degré/pixel
        
        # Pour garder trace
        self.processed = []
        self.ignored = []
    
    def find_files(self, obs_date, target=None):
        """Trouve les fichiers par type"""
        fits_dir = os.path.join(self.base_path, obs_date, "FITS")
        files = {'spectrum': [], 'tpi': [], 'image': []}
        
        for f in glob.glob(os.path.join(fits_dir, "*.fits")):
            match = re.match(self.file_pattern, os.path.basename(f))
            if match and (target is None or match.group(5) == target):
                ftype = match.group(3).lower()
                if ftype in files:
                    files[ftype].append({
                        'path': f,
                        'time': match.group(2),
                        'target': match.group(5)
                    })
        return files
    
    def process_spectrum(self, f):
        """
        Spectre : ON - OFF avec LEFT_POL + RIGHT_POL
        """
        with fits.open(f['path'], ignore_missing_end=True) as hdul:
            data = hdul[1].data if len(hdul) > 1 else hdul[0].data
            
            # Vérification rapide
            if 'LEFT_POL' not in data.dtype.names or 'STATUS' not in data.dtype.names:
                print(f"  ⚠️  {os.path.basename(f['path'])}: colonnes manquantes")
                return None
            
            # Extraire ON, OFF, CAL
            on_rows = data[data['STATUS'] == 'on']
            off_rows = data[data['STATUS'] == 'off']
            cal_rows = data[data['STATUS'] == 'cal']
            
            if len(on_rows) == 0 or len(off_rows) == 0:
                print(f"  ⚠️  {os.path.basename(f['path'])}: pas de ON/OFF")
                return None
            
            on = on_rows[0]
            off = off_rows[0]
            
            # Combinaison des polarisations
            spec_on = on['LEFT_POL'] + on['RIGHT_POL']
            spec_off = off['LEFT_POL'] + off['RIGHT_POL']
            spec_cal = spec_on - spec_off  # Spectre calibré
            
            # Fréquences
            freq = self.basefreq + np.linspace(0, self.bandwidth, self.nchan)
            
            # Plot
            fig = plt.figure(figsize=(12,5))  # <-- fig au lieu de plt            plt.plot(freq/1e6, spec_on, label='ON', alpha=0.7)
            plt.plot(freq/1e6, spec_off, label='OFF', alpha=0.7)
            plt.plot(freq/1e6, spec_cal, label='ON-OFF', linewidth=2)
            
            plt.xlabel("Fréquence (MHz)")
            plt.ylabel("Signal (ADU)")
            plt.title(f"{f['target']} - {f['time']}")
            plt.legend()
            plt.grid(True, alpha=0.3)
            
            return fig
        
    def sigma_clip_mask_robust(self, data, k=3):
        data = np.array(data)
        median = np.median(data)
        mad = np.median(np.abs(data - median))
        sigma = 1.4826 * mad  # conversion MAD -> sigma
        
        mask = np.abs(data - median) < k * sigma
        return mask
    
    def process_tpi(self, f, window=7, polyorder=2, start_ignore=1, end_ignore=0):
        """TPI : brut + RA/Dec + vitesses"""
        with fits.open(f['path']) as hdul:
            data = hdul[1].data if len(hdul) > 1 else hdul[0].data

            
            # Vérification que les données existent
            if len(data) == 0:
                print(f"  ⚠️  Fichier vide")
                return None
                    # Vérification des colonnes nécessaires
            required = ['JD', 'Azimuth', 'Elevation']
            for col in required:
                if col not in data.dtype.names:
                    print(f"  ⚠️  Colonne {col} manquante")
                    return None
                
            jd = data['JD']
            az = data['Azimuth']
            el = data['Elevation']
                
            # Vérification que jd n'est pas vide
            if len(jd) == 0:
                print(f"  ⚠️  Pas de données JD")
                return None
            
            # Filtrage temporel
            t = (jd - jd[0]) * 86400
            mask_time = (t > start_ignore) & (t < (t[-1] - end_ignore))
            jd = jd[mask_time]
            az = az[mask_time]
            el = el[mask_time]
            t = (jd - jd[0]) * 86400
            
            if len(data) == 0:
                return None
            
            # Conversion RA/Dec
            time = Time(jd, format='jd')
            altaz = AltAz(az=az*u.deg, alt=el*u.deg, 
                            location=self.location, obstime=time)
            sky = SkyCoord(altaz)
            ra_raw, dec_raw = sky.icrs.ra.deg, sky.icrs.dec.deg
            
            
            # Filtrage sigma-clip simple
            
            ra_mask = self.sigma_clip_mask_robust(ra_raw)
            dec_mask = self.sigma_clip_mask_robust(dec_raw)
            
            mask_comb = ra_mask & dec_mask
            
            if np.sum(mask_comb) < window:
                return {'raw': data, 'clean': data, 'ra': ra_raw, 'dec': dec_raw}
            
            # Vitesses
            dt = np.median(np.diff(t[mask_comb]))
            vit_ra = savgol_filter(ra_raw[mask_comb], window, polyorder, deriv=1, delta=dt)
            vit_dec = savgol_filter(dec_raw[mask_comb], window, polyorder, deriv=1, delta=dt)
            
            acc_ra = savgol_filter(ra_raw[mask_comb], window, polyorder, deriv=2, delta=dt)
            acc_dec = savgol_filter(dec_raw[mask_comb], window, polyorder, deriv=2, delta=dt)
            
            v_norm = np.sqrt(vit_ra**2 + vit_dec**2)
            a_norm = np.sqrt(acc_ra**2 + acc_dec**2)
            
            mask_vel = self.sigma_clip_mask_robust(v_norm, k=3)
            mask_acc = self.sigma_clip_mask_robust(a_norm, k=3)

            mask_va = mask_vel & mask_acc
            
            # data = data[mask_va]
            jd = jd[mask_comb][mask_va]
            az = az[mask_comb][mask_va]
            el = el[mask_comb][mask_va]
            
            # Conversion RA/Dec
            time = Time(jd, format='jd')
            altaz = AltAz(az=az*u.deg, alt=el*u.deg, 
                            location=self.location, obstime=time)
            sky = SkyCoord(altaz)
            ra_clean, dec_clean = sky.icrs.ra.deg, sky.icrs.dec.deg
            
            return {
                'raw': data,
                'clean': data[mask_time][mask_comb][mask_va],
                'ra': ra_clean,
                'dec': dec_clean,
                'vit_ra': vit_ra,
                'vit_dec': vit_dec,
                't': t
            }
        
            
    def get_image_resolution(self, obs_date, target):
        """
        Récupère la résolution d'une image FITS de la même cible
        """
        fits_dir = os.path.join(self.base_path, obs_date, "FITS")
        pattern = os.path.join(fits_dir, f"*_IMAGE-*-{target}_*.fits")
        
        image_files = glob.glob(pattern)
        if not image_files:
            return None
        
        try:
            with fits.open(image_files[0]) as hdul:
                for hdu in hdul:
                    if hdu.data is not None and len(hdu.data.shape) == 2:
                        return hdu.data.shape  # (ny, nx)
        except:
            pass
        return None
    
    def project_tpi_to_map(self, result, nx=51, ny=51, method='histogram2d', 
                            bbc_column=None, statistic='mean'):
        """
        Projette les données TPI sur une carte 2D
        
        Parameters:
        -----------
        result : dict
            Dictionnaire contenant 'ra', 'dec', et éventuellement 'clean' (tableau BBCs)
        nx, ny : int
            Dimensions de la carte
        method : str
            Méthode de projection: 'loop', 'histogram2d', 'binned_statistic'
        bbc_column : str
            Nom de la colonne BBC à projeter (ex: 'BBC1')
        statistic : str
            Statistique à calculer: 'mean', 'sum', 'count', 'std'
        
        Returns:
        --------
        dict : {'hit_map': carte de comptage, 'data_map': carte de données (si BBC), 
                'ra_edges', 'dec_edges', 'ra_center', 'dec_center'}
        """
        from scipy import stats
        
        ra = result['ra']
        dec = result['dec']
        
        # Définir les limites de la carte (avec marge de 5%)
        ra_min, ra_max = np.min(ra), np.max(ra)
        dec_min, dec_max = np.min(dec), np.max(dec)
        
        ra_margin = (ra_max - ra_min) * 0.05
        dec_margin = (dec_max - dec_min) * 0.05
        
        ra_edges = np.linspace(ra_min - ra_margin, ra_max + ra_margin, nx + 1)
        dec_edges = np.linspace(dec_min - dec_margin, dec_max + dec_margin, ny + 1)
        ra_center = (ra_edges[:-1] + ra_edges[1:]) / 2
        dec_center = (dec_edges[:-1] + dec_edges[1:]) / 2
        
        hit_map = np.zeros((ny, nx))
        data_map = np.zeros((ny, nx))
        
        # Méthode 1: Boucle simple
        if method == 'loop':
            for i in range(len(ra)):
                ix = np.digitize(ra[i], ra_edges) - 1
                iy = np.digitize(dec[i], dec_edges) - 1
                if 0 <= ix < nx and 0 <= iy < ny:
                    hit_map[iy, ix] += 1
                    if bbc_column and bbc_column in result['clean'].dtype.names:
                        data_map[iy, ix] += result['clean'][bbc_column][i]
            
            if bbc_column and np.sum(hit_map) > 0:
                data_map = np.divide(data_map, hit_map, where=hit_map>0)
        
        # Méthode 2: histogram2d (rapide pour les comptages)
        elif method == 'histogram2d':
            hit_map, _, _ = np.histogram2d(dec, ra, bins=[dec_edges, ra_edges])
            
            if bbc_column and bbc_column in result['clean'].dtype.names:
                data_sum, _, _ = np.histogram2d(dec, ra, bins=[dec_edges, ra_edges],
                                            weights=result['clean'][bbc_column])
                data_map = np.divide(data_sum, hit_map, where=hit_map>0)
        
        # Méthode 3: binned_statistic_2d (recommandée)
        elif method == 'binned_statistic':
            hit_map, _, _, _ = stats.binned_statistic_2d(
                dec, ra, None, statistic='count', bins=[dec_edges, ra_edges])
            
            if bbc_column and bbc_column in result['clean'].dtype.names:
                data_map, _, _, _ = stats.binned_statistic_2d(
                    dec, ra, result['clean'][bbc_column], 
                    statistic=statistic, bins=[dec_edges, ra_edges])
        
        return {
            'hit_map': hit_map,
            'data_map': data_map,
            'ra_edges': ra_edges,
            'dec_edges': dec_edges,
            'ra_center': ra_center,
            'dec_center': dec_center,
            'nx': nx, 'ny': ny
        }
    
    def plot_tpi_maps_comparison(self, maps_dict, target, time, out_dir, bbc_column=None):
        """
        Visualise les différentes cartes produites
        """
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        hit_map = maps_dict['hit_map']
        data_map = maps_dict['data_map']
        ra_center = maps_dict['ra_center']
        dec_center = maps_dict['dec_center']
        
        extent = [ra_center[0], ra_center[-1], dec_center[0], dec_center[-1]]
        
        # Carte des hits (échantillonnage)
        im0 = axes[0,0].imshow(hit_map, origin='lower', extent=extent, 
                            cmap='viridis', aspect='auto')
        axes[0,0].set_title(f"Hit map - {np.sum(hit_map>0)} pixels occupés")
        axes[0,0].set_xlabel("RA (deg)")
        axes[0,0].set_ylabel("Dec (deg)")
        axes[0,0].invert_xaxis()
        plt.colorbar(im0, ax=axes[0,0], label="Nombre d'échantillons")
        
        # Histogramme des hits
        if np.any(hit_map > 0):
            axes[0,1].hist(hit_map[hit_map > 0].flatten(), bins=50, alpha=0.7)
            axes[0,1].set_xlabel("Échantillons par pixel")
            axes[0,1].set_ylabel("Nombre de pixels")
            axes[0,1].set_title(f"Distribution échantillonnage")
            axes[0,1].axvline(np.mean(hit_map[hit_map>0]), color='r', 
                            ls='--', label=f"Moyenne: {np.mean(hit_map[hit_map>0]):.2f}")
            axes[0,1].legend()
        
        # Carte de données (si disponible)
        if np.any(data_map != 0):
            im2 = axes[0,2].imshow(data_map, origin='lower', extent=extent,
                                cmap='inferno', aspect='auto')
            axes[0,2].set_title(f"Carte {bbc_column if bbc_column else 'signal'}")
            axes[0,2].set_xlabel("RA (deg)")
            axes[0,2].set_ylabel("Dec (deg)")
            axes[0,2].invert_xaxis()
            plt.colorbar(im2, ax=axes[0,2], label="Signal (ADU)")
        
        # Cartes binaires (couverture)
        axes[1,0].imshow(hit_map > 0, origin='lower', extent=extent,
                        cmap='gray', aspect='auto')
        axes[1,0].set_title(f"Couverture binaire")
        axes[1,0].set_xlabel("RA (deg)")
        axes[1,0].set_ylabel("Dec (deg)")
        axes[1,0].invert_xaxis()
        
        # Statistiques
        stats_text = f"""
        Points totaux: {np.sum(hit_map):.0f}
        Pixels occupés: {np.sum(hit_map>0)}/{maps_dict['nx']*maps_dict['ny']}
        Taux couverture: {np.sum(hit_map>0)/(maps_dict['nx']*maps_dict['ny'])*100:.1f}%
        Max par pixel: {np.max(hit_map):.0f}
        """
        axes[1,1].text(0.1, 0.5, stats_text, transform=axes[1,1].transAxes,
                    fontsize=12, verticalalignment='center',
                    bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
        axes[1,1].axis('off')
        
        # Densité 2D (si scipy disponible)
        try:
            from scipy.stats import gaussian_kde
            if len(ra_center) > 10 and len(dec_center) > 10:
                # Échantillonner pour la KDE (trop dense)
                ra_sample = ra_center[::max(1, len(ra_center)//50)]
                dec_sample = dec_center[::max(1, len(dec_center)//50)]
                axes[1,2].remove()
                axes[1,2] = fig.add_subplot(2, 3, 6)
                axes[1,2].hist2d(ra_sample, dec_sample, bins=30, cmap='plasma')
                axes[1,2].set_xlabel("RA (deg)")
                axes[1,2].set_ylabel("Dec (deg)")
                axes[1,2].set_title("Distribution 2D")
                axes[1,2].invert_xaxis()
        except:
            axes[1,2].axis('off')
        
        plt.suptitle(f"{target} - {time} - Cartes d'échantillonnage")
        plt.tight_layout()
        
        out_path = os.path.join(out_dir, f"{target}_tpi_maps_{time}.png")
        fig.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"     💾 Cartes complètes: {out_path}")
    
    def study_resolution_effect(self, result, target, time, out_dir):
        """
        Étudie l'effet de la résolution sur la carte
        """
        resolutions = [(25,25), (51,51), (101,101), (201,201)]
        fig, axes = plt.subplots(2, 2, figsize=(14, 14))
        
        for ax, (res_x, res_y) in zip(axes.flat, resolutions):
            maps = self.project_tpi_to_map(result, nx=res_x, ny=res_y, 
                                        method='binned_statistic')
            im = ax.imshow(maps['hit_map'], origin='lower', cmap='viridis', aspect='auto')
            ax.set_title(f"{res_x}x{res_y} - {np.sum(maps['hit_map']>0)} pixels occupés")
            ax.set_xlabel("Pixel X")
            ax.set_ylabel("Pixel Y")
            plt.colorbar(im, ax=ax, fraction=0.046)
        
        plt.suptitle(f"{target} - {time} - Effet de la résolution")
        plt.tight_layout()
        
        out_path = os.path.join(out_dir, f"{target}_resolution_study_{time}.png")
        fig.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"     💾 Étude résolution: {out_path}")
    
    def compare_bbc_maps(self, result, target, time, out_dir, bbc_cols):
        """
        Compare les cartes des différentes BBCs
        """
        n_bbc = len(bbc_cols)
        n_cols = min(3, n_bbc)
        n_rows = (n_bbc + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
        axes = axes.flatten() if n_bbc > 1 else [axes]
        
        for i, bbc in enumerate(bbc_cols[:6]):  # Max 6 BBCs
            if i >= len(axes):
                break
                
            maps = self.project_tpi_to_map(result, method='binned_statistic',
                                        bbc_column=bbc, statistic='mean')
            
            if np.any(maps['data_map'] > 0):
                im = axes[i].imshow(maps['data_map'], origin='lower', 
                                    extent=[maps['ra_center'][0], maps['ra_center'][-1],
                                        maps['dec_center'][0], maps['dec_center'][-1]],
                                    cmap='inferno', aspect='auto')
                axes[i].set_title(f"{bbc}")
                axes[i].set_xlabel("RA (deg)")
                axes[i].set_ylabel("Dec (deg)")
                axes[i].invert_xaxis()
                plt.colorbar(im, ax=axes[i], fraction=0.046)
            else:
                axes[i].text(0.5, 0.5, f"Pas de données\npour {bbc}", 
                            ha='center', va='center', transform=axes[i].transAxes)
        
        # Cacher les axes vides
        for j in range(i+1, len(axes)):
            axes[j].axis('off')
        
        plt.suptitle(f"{target} - {time} - Comparaison des BBCs")
        plt.tight_layout()
        
        out_path = os.path.join(out_dir, f"{target}_bbc_comparison_{time}.png")
        fig.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"     💾 Comparaison BBCs: {out_path}")
    
    # [Votre méthode run existante, modifiée pour inclure les appels aux nouvelles méthodes]
    
    def run(self, obs_date, target=None, window=7, polyorder=2, start_ignore=1, end_ignore=0):
        """
        Lance le pipeline - version avec projection d'image
        """
        import matplotlib.pyplot as plt
        
        print(f"\n=== GRAD-300 {obs_date} {target or 'tout'} ===")
        
        files = self.find_files(obs_date, target)
        
        for t in ['spectrum', 'tpi', 'image']:
            print(f"{t}s: {len(files[t])}")
        
        # Création dossiers de sortie
        out_dir = os.path.join(self.base_path, obs_date, "output_nametarget")
        
        # ===== SPECTRES =====
        for f in files['spectrum']:
            print(f"\n📊 {f['target']} - {f['time']}")
            fig = self.process_spectrum(f)
            
            if fig:
                out = os.path.join(out_dir, "plots", f['target'], "unprocessed")
                os.makedirs(out, exist_ok=True)
                fig.savefig(os.path.join(out, f"{f['target']}_spectrum_{f['time']}.png"), dpi=150)
                plt.close(fig)
                self.processed.append(f"spectrum:{f['target']}:{f['time']}")
                print(f"  ✅ Sauvegardé")
            else:
                self.ignored.append(f"spectrum:{f['target']}:{f['time']}")
        
        # ===== TPI =====
        for f in files['tpi']:
            print(f"\n🎯 {f['target']} - {f['time']}")
            
            # Plot brut
            with fits.open(f['path']) as hdul:
                data = hdul[1].data if len(hdul) > 1 else hdul[0].data
                
                fig, ax = plt.subplots(figsize=(12,6))
                bbc_cols = [c for c in data.dtype.names if 'BBC' in c][:8]
                jd_col = data['JD']
                for col in bbc_cols:
                    ax.plot(jd_col, data[col], label=col, alpha=0.7, linewidth=1)
                ax.legend(loc='upper right', ncol=2)
                ax.set_xlabel("Time (JD)")
                ax.set_ylabel("Signal (ADU)")
                ax.set_title(f"TPI brut {f['target']} - {f['time']}")
                ax.grid(True, alpha=0.3)
                
                out_raw = os.path.join(out_dir, "plots", f['target'], "unprocessed")
                os.makedirs(out_raw, exist_ok=True)
                fig.savefig(os.path.join(out_raw, f"{f['target']}_tpi_{f['time']}.png"), dpi=150)
                plt.close(fig)
            
            # Plot traité
            result = self.process_tpi(f, window, polyorder, start_ignore, end_ignore)
            
            if result and 'vit_ra' in result:
                bbc_cols = [c for c in result['clean'].dtype.names if 'BBC' in c][:8]
                jd_col = result['clean']['JD']
                
                fig, axes = plt.subplots(2,2, figsize=(14,10))
                
                # RA/Dec
                axes[0,0].scatter(result['ra'], result['dec'], s=5, alpha=0.5, c='blue')
                axes[0,0].set_xlabel("RA (deg)")
                axes[0,0].set_ylabel("Dec (deg)")
                axes[0,0].set_title("RA/Dec")
                axes[0,0].invert_xaxis()
                axes[0,0].grid(True, alpha=0.3)
                
                # Vitesses
                axes[0,1].scatter(result['vit_ra'], result['vit_dec'], s=5, alpha=0.5, c='red')
                axes[0,1].set_xlabel("Vitesse RA (deg/s)")
                axes[0,1].set_ylabel("Vitesse Dec (deg/s)")
                axes[0,1].set_title("Vitesses")
                axes[0,1].grid(True, alpha=0.3)
                
                # RA/Dec coloré par vitesse
                sc = axes[1,0].scatter(result['ra'], result['dec'], s=5, cmap='viridis')
                axes[1,0].set_xlabel("RA (deg)")
                axes[1,0].set_ylabel("Dec (deg)")
                axes[1,0].set_title("RA/Dec (couleur = vitesse RA)")
                axes[1,0].invert_xaxis()
                axes[1,0].grid(True, alpha=0.3)
                plt.colorbar(sc, ax=axes[1,0])
                
                # BBCs nettoyées
                for col in bbc_cols:
                    axes[1,1].plot(jd_col, result['clean'][col], label=col, alpha=0.7, linewidth=1)
                axes[1,1].set_xlabel("Time (JD)")
                axes[1,1].set_ylabel("Signal (ADU)")
                axes[1,1].set_title(f"BBCs nettoyées - {f['target']}")
                axes[1,1].legend(loc='upper right', ncol=2, fontsize='small')
                axes[1,1].grid(True, alpha=0.3)
                
                fig.suptitle(f"{f['target']} - {f['time']} (window={window})")
                fig.tight_layout()
                
                out_proc = os.path.join(out_dir, "plots", f['target'], "processed")
                os.makedirs(out_proc, exist_ok=True)
                fig.savefig(os.path.join(out_proc, f"{f['target']}_tpi_processed_{f['time']}.png"), dpi=150)
                plt.close(fig)
                
                # ===== NOUVELLE PARTIE : RECONSTRUCTION D'IMAGE =====
                print(f"  --- Reconstruction d'image ---")
                
                # Récupérer la résolution des images FITS
                img_shape = self.get_image_resolution(obs_date, f['target'])
                
                if img_shape:
                    nx, ny = img_shape  # Note: nx = colonnes (RA), ny = lignes (Dec)
                    print(f"  Résolution image de référence: {nx} x {ny}")
                else:
                    nx, ny = 51, 51  # Résolution par défaut
                    print(f"  Pas d'image de référence, utilisation {nx} x {ny}")
                
                # Créer le dossier pour les cartes
                maps_dir = os.path.join(out_dir, "maps", f['target'])
                os.makedirs(maps_dir, exist_ok=True)
                
                # 1. Carte simple avec la méthode actuelle (hit map)
                # (c'est celle que vous avez déjà codée - je la garde)
                
                # 2. Tester différentes méthodes de projection
                print(f"  Test des méthodes de projection...")
                methods = ['loop', 'histogram2d', 'binned_statistic']
                
                for method in methods:
                    # Carte de hits seulement
                    maps_hit = self.project_tpi_to_map(result, nx=nx, ny=ny, 
                                                    method=method)
                    
                    # Sauvegarder la hit map
                    np.save(os.path.join(maps_dir, f"{f['target']}_hitmap_{method}_{f['time']}.npy"), 
                        maps_hit['hit_map'])
                    
                    # Pour la première BBC, créer aussi une carte de données
                    if bbc_cols:
                        bbc = bbc_cols[0]
                        maps_data = self.project_tpi_to_map(result, nx=nx, ny=ny,
                                                            method=method,
                                                            bbc_column=bbc,
                                                            statistic='mean')
                        np.save(os.path.join(maps_dir, f"{f['target']}_{bbc}_{method}_{f['time']}.npy"),
                            maps_data['data_map'])
                
                # 3. Visualisation complète avec la meilleure méthode
                print(f"  Création des visualisations...")
                maps_best = self.project_tpi_to_map(result, nx=nx, ny=ny,
                                                    method='binned_statistic')
                self.plot_tpi_maps_comparison(maps_best, f['target'], f['time'], 
                                            os.path.join(out_dir, "plots", f['target'], "processed"))
                
                # 4. Étude de l'effet de résolution
                self.study_resolution_effect(result, f['target'], f['time'],
                                            os.path.join(out_dir, "plots", f['target'], "processed"))
                
                # 5. Comparaison des BBCs (si plusieurs)
                if len(bbc_cols) >= 2:
                    self.compare_bbc_maps(result, f['target'], f['time'],
                                        os.path.join(out_dir, "plots", f['target'], "processed"),
                                        bbc_cols)
                
                # 6. Option: moyenne des BBCs pour simuler RIGHT_POL
                if len(bbc_cols) >= 2:
                    # Créer un signal moyen des BBCs
                    mean_signal = np.mean([result['clean'][bbc] for bbc in bbc_cols], axis=0)
                    
                    # Ajouter temporairement au result pour projection
                    result_temp = result.copy()
                    # Note: il faudrait modifier la structure, mais c'est une idée
                
                print(f"  ✅ Reconstruction d'image terminée")
                self.processed.append(f"tpi_map:{f['target']}:{f['time']}")
            
            else:
                self.ignored.append(f"tpi:{f['target']}:{f['time']}")
                print(f"  ⚠️  Ignoré (pas assez de données)")
        
        # ===== IMAGES =====
        for f in files['image']:
            print(f"\n🖼️  {f['target']} - {f['time']}")
            
            try:
                with fits.open(f['path']) as hdul:
                    data = None
                    for hdu in hdul:
                        if hdu.data is not None and len(hdu.data.shape) >= 2:
                            data = hdu.data
                            break
                    
                    if data is None:
                        self.ignored.append(f"image:{f['target']}:{f['time']}")
                        continue
                    
                    fig, ax = plt.subplots(figsize=(10,8))
                    if len(data.shape) == 2:
                        im = ax.imshow(data, cmap='viridis', origin='lower', aspect='auto')
                        plt.colorbar(im, ax=ax, label='Intensité')
                    else:
                        ax.plot(data.flatten())
                    
                    ax.set_title(f"Image {f['target']} - {f['time']}")
                    
                    out_img = os.path.join(out_dir, "plots", f['target'], "unprocessed")
                    os.makedirs(out_img, exist_ok=True)
                    fig.savefig(os.path.join(out_img, f"{f['target']}_image_{f['time']}.png"), dpi=150)
                    plt.close(fig)
                    
                    self.processed.append(f"image:{f['target']}:{f['time']}")
                    print(f"  ✅ Sauvegardé")
                    
            except Exception as e:
                print(f"  ❌ Erreur: {e}")
                self.ignored.append(f"image:{f['target']}:{f['time']}")
        
        print(f"\n✅ Terminé: {len(self.processed)} traités, {len(self.ignored)} ignorés")
        return self.processed
# Utilisation
if __name__ == "__main__":
    pipe = GradPipeline()
    pipe.run("20250212", "SUN", window=9)


=== GRAD-300 20250212 SUN ===
spectrums: 1
tpis: 4
images: 4

📊 SUN - 083051
  ✅ Sauvegardé

🎯 SUN - 143921


DIAMETER =                    3  / TELESCOPE diameter                            [astropy.io.fits.card]
DIAMETER =                    0  / TELESCOPE diameter                            [astropy.io.fits.card]
/tmp/ipykernel_173101/770751997.py:530: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  sc = axes[1,0].scatter(result['ra'], result['dec'], s=5, cmap='viridis')


  --- Reconstruction d'image ---
  Résolution image de référence: 51 x 51
  Test des méthodes de projection...
  Création des visualisations...


DIAMETER =                     0  / TELESCOPE diameter                           [astropy.io.fits.card]
/tmp/ipykernel_173101/770751997.py:262: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  data_map = np.divide(data_map, hit_map, where=hit_map>0)
/tmp/ipykernel_173101/770751997.py:271: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  data_map = np.divide(data_sum, hit_map, where=hit_map>0)


     💾 Cartes complètes: ./GRAD-300/GRAD-300/20250212/output_nametarget/plots/SUN/processed/SUN_tpi_maps_143921.png
     💾 Étude résolution: ./GRAD-300/GRAD-300/20250212/output_nametarget/plots/SUN/processed/SUN_resolution_study_143921.png
     💾 Comparaison BBCs: ./GRAD-300/GRAD-300/20250212/output_nametarget/plots/SUN/processed/SUN_bbc_comparison_143921.png
  ✅ Reconstruction d'image terminée

🎯 SUN - 083213


  --- Reconstruction d'image ---
  Résolution image de référence: 51 x 51
  Test des méthodes de projection...
  Création des visualisations...
     💾 Cartes complètes: ./GRAD-300/GRAD-300/20250212/output_nametarget/plots/SUN/processed/SUN_tpi_maps_083213.png
     💾 Étude résolution: ./GRAD-300/GRAD-300/20250212/output_nametarget/plots/SUN/processed/SUN_resolution_study_083213.png
     💾 Comparaison BBCs: ./GRAD-300/GRAD-300/20250212/output_nametarget/plots/SUN/processed/SUN_bbc_comparison_083213.png
  ✅ Reconstruction d'image terminée

🎯 SUN - 105736


  --- Reconstruction d'image ---
  Résolution image de référence: 51 x 51
  Test des méthodes de projection...
  Création des visualisations...
     💾 Cartes complètes: ./GRAD-300/GRAD-300/20250212/output_nametarget/plots/SUN/processed/SUN_tpi_maps_105736.png
     💾 Étude résolution: ./GRAD-300/GRAD-300/20250212/output_nametarget/plots/SUN/processed/SUN_resolution_study_105736.png
     💾 Comparaison BBCs: ./GRAD-300/GRAD-300/20250212/output_nametarget/plots/SUN/processed/SUN_bbc_comparison_105736.png
  ✅ Reconstruction d'image terminée

🎯 SUN - 090549


  --- Reconstruction d'image ---
  Résolution image de référence: 51 x 51
  Test des méthodes de projection...
  Création des visualisations...
     💾 Cartes complètes: ./GRAD-300/GRAD-300/20250212/output_nametarget/plots/SUN/processed/SUN_tpi_maps_090549.png
     💾 Étude résolution: ./GRAD-300/GRAD-300/20250212/output_nametarget/plots/SUN/processed/SUN_resolution_study_090549.png
     💾 Comparaison BBCs: ./GRAD-300/GRAD-300/20250212/output_nametarget/plots/SUN/processed/SUN_bbc_comparison_090549.png
  ✅ Reconstruction d'image terminée

🖼️  SUN - 090552
  ✅ Sauvegardé

🖼️  SUN - 143918
  ✅ Sauvegardé

🖼️  SUN - 083216


  ✅ Sauvegardé

🖼️  SUN - 105738
  ✅ Sauvegardé

✅ Terminé: 9 traités, 0 ignorés


In [ ]:
# reconstituer limage après cleaning 
# mettre le temps en secondes dans le spectrum ? 
# remplacer le 3eme plot par qqch de plus parlant (acceleration ra dec ?)